# 05 · Challenger models (H4, H5)

Could a different approach map deprivation more fairly? I train four models on a simulated household survey
drawn from the census and score them on the localities the survey never visited, separately for Indigenous
and non-Indigenous places. Each model predicts a locality's infrastructure-deprivation rate: gradient
boosting on night lights; gradient boosting on the satellite embeddings; the same model with municipality
effects estimated from the survey; and a Fay-Herriot small area estimation model, of the kind statistical
offices use, on the embeddings' principal components. Because the census records the true rate everywhere,
every prediction can be scored exactly.

**Runtime:** 25-50 minutes (50 survey replications). **Requires:** the satellite features from notebook 04.

In [ ]:
# Make the project importable; works in JupyterLab, VS Code and Colab without reinstalling.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from mapaudit import notebook as nb
from mapaudit import pipeline, report
from mapaudit.config import get_paths

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)
MODE = nb.notebook_mode()  # "real" by default; CI sets MAPAUDIT_NOTEBOOK_MODE=demo (synthetic data)
print("Mode:", MODE)

In [ ]:
if MODE == "demo":
    nb.prepare_demo(through="build")
info = pipeline.simulate(MODE)
report.report(MODE)
info

## How the models are scored

Fixed in the pre-analysis plan before any real data were seen:

- **RMSE** of the predicted deprivation rate is the accuracy measure, because the budget formula adds up
  predicted rates and squared error is the loss whose best prediction is the mean.
- **Signed bias** is the fairness measure: does a model systematically understate deprivation for a group?
- **Calibration slope** (above 1 = predictions squeezed toward the average) and **variance ratio** (below 1 =
  the same) show whether a low RMSE was bought by compressing the extremes, where programs act.
- Everything is computed **per person** (population-weighted, primary) and **per community** (unweighted).
- No MAPE or MSLE: deprivation rates are often zero, and both measures discount errors in the poorest places.

## H4 · Which model maps deprivation best, and for whom?

In [ ]:
nb.show_figure("challenger_bias", MODE)
metrics = nb.load_table("challenger_metrics_summary", MODE)
people = metrics[metrics["weighting"] == "people"]
people.pivot(index="model", columns="group", values=["bias_mean", "rmse_mean", "calibration_slope_mean"]).round(3)

H4 as pre-registered: the difference in absolute bias for Indigenous localities between gradient boosting and
small area estimation, computed on the same simulated survey in each replication, so the luck of the draw
cancels out. Supported if the 95% replication range lies above zero. The other rows compare each model's
RMSE with small area estimation the same way.

In [ ]:
paired = nb.load_table("challenger_paired", MODE)
paired["hypothesis"] = paired["hypothesis"].fillna("")
paired["supported"] = paired["supported"].fillna("")
paired.round(4)

## Where the errors fall: bias by true deprivation decile

Compression shows up as positive bias in the least deprived deciles and negative bias in the most deprived
ones. Per-community metrics weight each locality equally, which matters because Indigenous localities are
many and small.

In [ ]:
nb.show_figure("challenger_deciles", MODE)
places = metrics[metrics["weighting"] == "places"]
places.pivot(index="model", columns="group", values=["bias_mean", "rmse_mean"]).round(3)

## H5 · Are the 90% intervals honest for everyone?

"Survey frame" = localities large enough for the simulated survey to visit. Conformal guarantees only hold
where calibration and prediction look alike. The interval score adds width and a penalty for every miss, so
it rewards intervals that are both narrow and honest (lower is better).

In [ ]:
nb.show_figure("conformal_coverage", MODE)
coverage = nb.load_table("challenger_coverage_summary", MODE)
coverage[coverage["group"] != "All"].pivot_table(
    index=["model", "method"], columns=["frame", "group"], values=["coverage_mean", "interval_score_mean"]
).round(3)

## Targeting and money

The allocation rules from notebook 03, applied to each model's predictions: the share of the truly poorest
left out when the predicted poorest 20% are targeted, and how much of the municipal budget is misallocated,
in total and for the median municipality.

In [ ]:
targeting = nb.load_table("challenger_targeting_summary", MODE)
display(targeting.pivot(index="model", columns="group", values="exclusion_rate_mean").round(3))
nb.show_figure("allocation", MODE)
nb.load_table("challenger_municipal_summary", MODE).round(3)

Who these comparisons cover (localities with suppressed census data cannot be scored):

In [ ]:
nb.load_table("challenger_evaluation_coverage", MODE).round(3)